In [1]:
import numpy as np
import pandas as pd
import matplotlib.colors as mcolors
import matplotlib.pyplot as plt
import time

from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV
from sklearn.calibration import CalibratedClassifierCV
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import balanced_accuracy_score

import mlacca_functions as mlcf

In [2]:
df_path = 'data\\csv_files\\new_vk_areas_df.csv'
df_with_pahs_path = 'data\\csv_files\\new_vk_areas_with_pahs.csv'

weighted = True

In [3]:
df = pd.read_csv(df_path)
df_with_pahs = pd.read_csv(df_with_pahs_path)

In [4]:
columns_2d = ['O/C','H/C']
X_2d = df[columns_2d].to_numpy()

columns_3d = ['O/C','H/C', 'N/C']
X_3d = df[columns_3d].to_numpy()

y = df['category'].to_numpy()
weights = df['weights'].to_numpy() if weighted else None


X_2d_with_pahs = df_with_pahs[columns_2d].to_numpy()
X_3d_with_pahs = df_with_pahs[columns_3d].to_numpy()

y_with_pahs = df_with_pahs['category'].to_numpy()

weights_with_pahs = df_with_pahs['weights'].to_numpy() if weighted else None

## Optimise KNN

In [5]:
param_grid_knn = {
    'n_neighbors': [1,5,10,50,100,150,200],
    }

In [6]:
# X_2d_scaled = StandardScaler().fit_transform(X_2d)

# grid_knn_2d = GridSearchCV(KNeighborsClassifier(weights='distance'),
#                     param_grid_knn, refit=True, verbose=3, scoring='balanced_accuracy')

# grid_knn_2d.fit(X_2d_scaled, y)

# print('2D KNN best_estimator_:', grid_knn_2d.best_estimator_)

In [7]:
X_3d_scaled = StandardScaler().fit_transform(X_3d)

grid_knn_3d = GridSearchCV(KNeighborsClassifier(weights='distance'),
                    param_grid_knn, refit=True, verbose=3, scoring='balanced_accuracy') 

grid_knn_3d.fit(X_3d_scaled, y)

print('3D KNN best_estimator_:', grid_knn_3d.best_estimator_)

Fitting 5 folds for each of 7 candidates, totalling 35 fits
[CV 1/5] END .....................n_neighbors=1;, score=0.920 total time=   0.0s
[CV 2/5] END .....................n_neighbors=1;, score=0.831 total time=   0.0s
[CV 3/5] END .....................n_neighbors=1;, score=0.887 total time=   0.0s
[CV 4/5] END .....................n_neighbors=1;, score=0.676 total time=   0.0s
[CV 5/5] END .....................n_neighbors=1;, score=0.762 total time=   0.0s
[CV 1/5] END .....................n_neighbors=5;, score=0.915 total time=   0.0s
[CV 2/5] END .....................n_neighbors=5;, score=0.903 total time=   0.0s
[CV 3/5] END .....................n_neighbors=5;, score=0.891 total time=   0.0s
[CV 4/5] END .....................n_neighbors=5;, score=0.696 total time=   0.0s
[CV 5/5] END .....................n_neighbors=5;, score=0.806 total time=   0.0s
[CV 1/5] END ....................n_neighbors=10;, score=0.949 total time=   0.0s
[CV 2/5] END ....................n_neighbors=10;,

In [8]:
# X_2d_with_pahs_scaled = StandardScaler().fit_transform(X_2d_with_pahs)

# grid_knn_pahs_2d = GridSearchCV(KNeighborsClassifier(weights='distance'),
#                     param_grid_knn, refit=True, verbose=3, scoring='balanced_accuracy') 

# grid_knn_pahs_2d.fit(X_2d_with_pahs_scaled, y_with_pahs)

# print('2D KNN with PAHs best_estimator_:', grid_knn_pahs_2d.best_estimator_)

In [9]:
X_3d_with_pahs_scaled = StandardScaler().fit_transform(X_3d_with_pahs)

grid_knn_pahs_3d = GridSearchCV(KNeighborsClassifier(weights='distance'),
                    param_grid_knn, refit=True, verbose=3, scoring='balanced_accuracy') 

grid_knn_pahs_3d.fit(X_3d_with_pahs_scaled, y_with_pahs)

print('3D KNN with PAHs best_params_:', grid_knn_pahs_3d.best_params_) 
print('3D KNN with PAHs best_estimator_:', grid_knn_pahs_3d.best_estimator_)

Fitting 5 folds for each of 7 candidates, totalling 35 fits
[CV 1/5] END .....................n_neighbors=1;, score=0.764 total time=   0.0s
[CV 2/5] END .....................n_neighbors=1;, score=0.804 total time=   0.0s
[CV 3/5] END .....................n_neighbors=1;, score=0.827 total time=   0.0s
[CV 4/5] END .....................n_neighbors=1;, score=0.915 total time=   0.0s
[CV 5/5] END .....................n_neighbors=1;, score=0.723 total time=   0.0s
[CV 1/5] END .....................n_neighbors=5;, score=0.832 total time=   0.0s
[CV 2/5] END .....................n_neighbors=5;, score=0.870 total time=   0.0s
[CV 3/5] END .....................n_neighbors=5;, score=0.856 total time=   0.0s
[CV 4/5] END .....................n_neighbors=5;, score=0.942 total time=   0.0s
[CV 5/5] END .....................n_neighbors=5;, score=0.792 total time=   0.0s
[CV 1/5] END ....................n_neighbors=10;, score=0.850 total time=   0.0s
[CV 2/5] END ....................n_neighbors=10;,

## Optimising RBF SVC
The usual `GridSearchCV` doesn't seem to work properly for RBF: it outputs values of 100,000 for C, which is patently nonsensical for a dataset of <10,000 observations. Therefore, another – unfortunately slower – approach is taken.

In [10]:
param_grid_svm_rbf = {
    'C': [10**(x) for x in range(5)],
    'gamma': ['auto'] + [10**(-x) for x in range(5)],
    }
param_grid_svm_rbf

{'C': [1, 10, 100, 1000, 10000],
 'gamma': ['auto', 1, 0.1, 0.01, 0.001, 0.0001]}

In [11]:
def rbf_opt(param_grid,X,y,repeats=40,weighted=True):
    param_acc = {f"C={c}, gamma={g}":[] for g in param_grid['gamma'] for c in param_grid['C']}

    for n in range(repeats):

        X_rdm_train, X_rdm_test, y_rdm_train, y_rdm_test, \
        weights_rdm_train, weights_rdm_test = mlcf.train_test(X, y, random_state=None)


        for c in param_grid['C']:
            for g in param_grid['gamma']:
                print(f'{n+1}/{repeats}\tTesting parameters: C={c}, gamma={g}')

                fit_args = dict(X=X_rdm_train, y=y_rdm_train)
                if weighted: fit_args['sample_weight'] = weights_rdm_train

                y_pred = SVC(kernel='rbf',C=c,gamma=g).fit(**fit_args).predict(X_rdm_test)

                param_acc[f'C={c}, gamma={g}'].append(balanced_accuracy_score(y_true=y_rdm_test,
                                                                              y_pred=y_pred,
                                                                              sample_weight=weights_rdm_test if weighted else None))
                print(f'\tBalanced accuracy = {param_acc[f'C={c}, gamma={g}'][-1]}')
    
    for combo in param_acc:
        param_acc[combo] = np.mean(param_acc[combo])
    
    return list(param_acc.keys())[np.argmax(np.array(list(param_acc.values())).flatten())]

In [12]:
# print('2D SVM (RBF) best_estimator_:', rbf_opt(param_grid_svm_rbf,X_2d,y))

In [13]:
print('3D SVM (RBF) best_estimator_:', rbf_opt(param_grid_svm_rbf,X_3d,y))

1/40	Testing parameters: C=1, gamma=auto
	Balanced accuracy = 0.927713339716893
1/40	Testing parameters: C=1, gamma=1
	Balanced accuracy = 0.9107309735138208
1/40	Testing parameters: C=1, gamma=0.1
	Balanced accuracy = 0.9085159552283836
1/40	Testing parameters: C=1, gamma=0.01
	Balanced accuracy = 0.819573895108095
1/40	Testing parameters: C=1, gamma=0.001
	Balanced accuracy = 0.4333333333333333
1/40	Testing parameters: C=1, gamma=0.0001
	Balanced accuracy = 0.16666666666666666
1/40	Testing parameters: C=10, gamma=auto
	Balanced accuracy = 0.9326171825428871
1/40	Testing parameters: C=10, gamma=1
	Balanced accuracy = 0.9212745022346897
1/40	Testing parameters: C=10, gamma=0.1
	Balanced accuracy = 0.9273603211776512
1/40	Testing parameters: C=10, gamma=0.01
	Balanced accuracy = 0.9086801588408631
1/40	Testing parameters: C=10, gamma=0.001
	Balanced accuracy = 0.819573895108095
1/40	Testing parameters: C=10, gamma=0.0001
	Balanced accuracy = 0.49075533661740556
1/40	Testing parameters: 

In [14]:
# print('2D SVM (RBF) with PAHs best_estimator_', rbf_opt(param_grid_svm_rbf,X_2d_with_pahs,y_with_pahs))

In [15]:
print('3D SVM (RBF) with PAHs best_estimator_', rbf_opt(param_grid_svm_rbf,X_3d_with_pahs,y_with_pahs))

1/40	Testing parameters: C=1, gamma=auto
	Balanced accuracy = 0.9412548128306272
1/40	Testing parameters: C=1, gamma=1
	Balanced accuracy = 0.9289592895947757
1/40	Testing parameters: C=1, gamma=0.1
	Balanced accuracy = 0.9247098184048576
1/40	Testing parameters: C=1, gamma=0.01
	Balanced accuracy = 0.8245983415015941
1/40	Testing parameters: C=1, gamma=0.001
	Balanced accuracy = 0.5152414928834671
1/40	Testing parameters: C=1, gamma=0.0001
	Balanced accuracy = 0.13999999999999999
1/40	Testing parameters: C=10, gamma=auto
	Balanced accuracy = 0.941795110490298
1/40	Testing parameters: C=10, gamma=1
	Balanced accuracy = 0.951786590183567
1/40	Testing parameters: C=10, gamma=0.1
	Balanced accuracy = 0.9300953304922481
1/40	Testing parameters: C=10, gamma=0.01
	Balanced accuracy = 0.9247098184048576
1/40	Testing parameters: C=10, gamma=0.001
	Balanced accuracy = 0.8245983415015941
1/40	Testing parameters: C=10, gamma=0.0001
	Balanced accuracy = 0.5229059833670971
1/40	Testing parameters: 

## Optimising Polynomial SVC

In [16]:
# choose C and gamma hyperparameters for SVM with polynomial kernel for the 3D case
# https://www.geeksforgeeks.org/machine-learning/svm-hyperparameter-tuning-using-gridsearchcv-ml/

param_grid_svm_poly = {
    'C': [10**x for x in range(4)],
    'gamma': ['auto'] + [10**(-x) for x in range(5)],
    'degree': list(range(1,8)),
    }
param_grid_svm_poly

{'C': [1, 10, 100, 1000],
 'gamma': ['auto', 1, 0.1, 0.01, 0.001, 0.0001],
 'degree': [1, 2, 3, 4, 5, 6, 7]}

In [17]:
# grid_poly_svm_2d = GridSearchCV(SVC(kernel='poly'),
#                                 param_grid_svm_poly, refit=True, verbose=3, scoring='balanced_accuracy') 

# grid_poly_svm_2d.fit(X_2d, y, sample_weight=weights if weighted else None)

# print('2D SVM (poly) best_estimator_:', grid_poly_svm_2d.best_estimator_)

In [18]:
grid_poly_svm_3d = GridSearchCV(SVC(kernel='poly'),
                    param_grid_svm_poly, refit=True, verbose=3, scoring='balanced_accuracy') 

grid_poly_svm_3d.fit(X_3d, y, sample_weight=weights if weighted else None)

print('3D SVM (poly) best_estimator_:', grid_poly_svm_3d.best_estimator_)

Fitting 5 folds for each of 168 candidates, totalling 840 fits
[CV 1/5] END .........C=1, degree=1, gamma=auto;, score=0.955 total time=   0.3s
[CV 2/5] END .........C=1, degree=1, gamma=auto;, score=0.921 total time=   0.3s
[CV 3/5] END .........C=1, degree=1, gamma=auto;, score=0.922 total time=   0.3s
[CV 4/5] END .........C=1, degree=1, gamma=auto;, score=0.825 total time=   0.2s
[CV 5/5] END .........C=1, degree=1, gamma=auto;, score=0.852 total time=   0.2s
[CV 1/5] END ............C=1, degree=1, gamma=1;, score=0.954 total time=   0.2s
[CV 2/5] END ............C=1, degree=1, gamma=1;, score=0.922 total time=   0.2s
[CV 3/5] END ............C=1, degree=1, gamma=1;, score=0.884 total time=   0.2s
[CV 4/5] END ............C=1, degree=1, gamma=1;, score=0.768 total time=   0.1s
[CV 5/5] END ............C=1, degree=1, gamma=1;, score=0.864 total time=   0.2s
[CV 1/5] END ..........C=1, degree=1, gamma=0.1;, score=0.939 total time=   0.5s
[CV 2/5] END ..........C=1, degree=1, gamma=0.

In [19]:
# grid_poly_svm_pahs_2d = GridSearchCV(SVC(kernel='poly',),
#                     param_grid_svm_poly, refit=True, verbose=3, scoring='balanced_accuracy') 

# grid_poly_svm_pahs_2d.fit(X_2d_with_pahs, y_with_pahs, sample_weight=weights_with_pahs if weighted else None)

# print('2D SVM (poly) with PAHs best_estimator_:', grid_poly_svm_pahs_2d.best_estimator_)

In [20]:
grid_poly_svm_pahs_3d = GridSearchCV(SVC(kernel='poly',),
                    param_grid_svm_poly, refit=True, verbose=3, scoring='balanced_accuracy') 

grid_poly_svm_pahs_3d.fit(X_3d_with_pahs, y_with_pahs, sample_weight=weights_with_pahs if weighted else None)

print('3D SVM (poly) with PAHs best_estimator_:', grid_poly_svm_pahs_3d.best_estimator_)

Fitting 5 folds for each of 168 candidates, totalling 840 fits
[CV 1/5] END .........C=1, degree=1, gamma=auto;, score=0.932 total time=   0.3s
[CV 2/5] END .........C=1, degree=1, gamma=auto;, score=0.894 total time=   0.3s
[CV 3/5] END .........C=1, degree=1, gamma=auto;, score=0.962 total time=   0.5s
[CV 4/5] END .........C=1, degree=1, gamma=auto;, score=0.887 total time=   0.5s
[CV 5/5] END .........C=1, degree=1, gamma=auto;, score=0.916 total time=   0.4s
[CV 1/5] END ............C=1, degree=1, gamma=1;, score=0.916 total time=   0.3s
[CV 2/5] END ............C=1, degree=1, gamma=1;, score=0.886 total time=   0.2s
[CV 3/5] END ............C=1, degree=1, gamma=1;, score=0.954 total time=   0.3s
[CV 4/5] END ............C=1, degree=1, gamma=1;, score=0.898 total time=   0.4s
[CV 5/5] END ............C=1, degree=1, gamma=1;, score=0.920 total time=   0.2s
[CV 1/5] END ..........C=1, degree=1, gamma=0.1;, score=0.905 total time=   0.8s
[CV 2/5] END ..........C=1, degree=1, gamma=0.